# H-001 · Core universe bake-off

**Decides:** `UNIVERSE_STAR` from a short research-IS TSMOM bake-off of **A** (FX_G10) vs **B** (ETF_MULTI).

**Does not decide:** `HORIZON_STAR`, vol targeting, family, overlays, sealed holdout.

Rank on median fold-val **net** Sharpe of the 12M probe only. 1M / 3M are robustness plots. Do **not** open 2017–2019.

Sleeve: **core**. Primary metric: **net** `ann_sharpe`. `psr` = P(true SR > 0). STAR is manual after the 12M table.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s3_trend.report import (
    arm_selection_table,
    fold_table,
    median_sharpe_hint,
)
from backtest.s3_trend.research import (
    RESEARCH_IS_END_S3,
    config_from_stack,
    n_trials_local,
    n_trials_stack,
    register_hypothesis_arms,
    split_is_oos,
    star_stack_path,
)
from backtest.s3_trend.runner import run_s3_backtest
from backtest.s3_trend.walkforward import embargo_bars_for_config, make_s3_folds
from backtest.star_stack_io import load_star_stack, update_star_stack_key
from data.processing.s3_price_panel import (
    S3_HOLDOUT_END,
    build_s3_price_panel,
    core_universe,
    price_panel_path,
    s3_data_dir,
)
from strategies.s3_trend.metrics import PERIODS_PER_YEAR, metrics_from_returns

DATA_DIR = s3_data_dir(ROOT)
ARTIFACTS = os.path.join(ROOT, "04_backtest", "s3_trend", "artifacts")
os.makedirs(ARTIFACTS, exist_ok=True)
SLEEVE = "core"
HYP_ID = "H-001"
STACK_PATH = star_stack_path(SLEEVE)
IS_START = pd.Timestamp("2007-01-01")
UNIVERSES = ("A", "B")
PROBES = {"12m": "single_12m", "3m": "single_3m", "1m": "single_1m"}
RANK_PROBE = "12m"
FULL_IS_COLS = [
    "arm",
    "ann_sharpe",
    "ann_sharpe_gross",
    "max_drawdown",
    "corr_to_s1",
    "corr_to_s2",
    "calmar",
    "psr",
    "dsr_local",
    "dsr_stack",
    "skew",
    "excess_kurtosis",
    "cost_bps_per_year",
    "n_days",
    "n_entries",
]
print("ROOT=", ROOT)
print("DATA_DIR=", DATA_DIR)
print("RESEARCH_IS_END_S3=", RESEARCH_IS_END_S3)
print("S3_HOLDOUT_END=", S3_HOLDOUT_END)


## 1. Calendar — IS / OOS splits

Same cut for both books. Session counts differ because FX uses the NY 17:00 day and ETFs use the cash session — those dates are printed in §4 after each panel loads.

| Window | Dates | Role |
|---|---|---|
| Research IS | 2007-01-01 through **2016-12-31** | Only window used to pick `UNIVERSE_STAR` |
| Sealed holdout | 2017-01-01 through 2019-12-31 | Shown, **not scored** |
| 2020+ | after 2019-12-31 | Later regime check; never a STAR input |

**IS:OOS = 10:3** (~77% / 23% of the 2007–2019 window). 2020+ is excluded from the ratio.


In [ ]:
print("Research IS: 2007-01-01 through", RESEARCH_IS_END_S3.date(), "(10 calendar years)")
print("Sealed holdout: 2017-01-01 through", S3_HOLDOUT_END.date(), "(3 calendar years)")
print("IS:OOS = 10:3 (~77% / 23% of the 2007-2019 window)")
print("2020+ excluded from the ratio (regime check, never a STAR decision)")


## 2. Membership and costs

- **Universe** = the book (A or B). Not a strategy arm.
- **A / FX_G10** — seven USD pairs, cost profile `A_FX_OANDA_S3` (spread + 0.1 pip slippage + swap), fill at the next open after the NY 17:00 close.
- **B / ETF_MULTI** — 19 equity / bond / commodity / USD ETFs, cost profile `B_ETF_ALPACA` (0 commission, ~3.3 bps/leg open-fill), fill at the next cash-session open. No FX 17:00 clock on this book.

`config_from_stack` maps pairs and costs from `UNIVERSE_STAR`. The event sleeve ignores this key and stays Forex.


In [ ]:
rows = []
for label in UNIVERSES:
    uni = core_universe(label)
    names = list(uni["instruments"])
    cfg = config_from_stack({"UNIVERSE_STAR": label})
    rows.append(
        {
            "universe": label,
            "name": uni["name"],
            "kind": uni["kind"],
            "n": len(names),
            "cost_profile": cfg.cost_profile,
            "include_swap": cfg.include_swap,
            "instruments": ",".join(names),
        }
    )
display(pd.DataFrame(rows))


## 3. Probe arms

An **arm** here is a TSMOM recipe used as a measuring stick. It is not a universe. Factorial keys are `{universe}_{probe}` (e.g. `A_12m`). Only the 12M probe freezes `UNIVERSE_STAR`. Do **not** write `HORIZON_STAR` from 1M / 3M — that freeze is H-003.

### Ranking probe (only this freezes the universe)

- **`tsmom_12m`** — Moskowitz / Asness / Pedersen 12-month time-series momentum. Sign of ~252-day return, ±1, no instrument/book vol target, no conviction scaling, overlays off. This is the `S3SimConfig` default. Same rule on both books, each with its own costs and fill calendar.

### Robustness only (plot them; do not rank on them; do not write `HORIZON_STAR`)

- **`tsmom_3m`** — ~63-day lookback. Faster, more turnover, more cost-sensitive. Ask: “does the universe ranking flip?”
- **`tsmom_1m`** — ~21-day. Diagnostic for “trend is just noise + costs.” Expected to look worse after costs, especially FX swap / ETF open slippage.

Not in this notebook: `equal_blend` (H-003), Donchian / family (H-007), pair or book vol target (H-004 / H-005), long-only ETF diagnostic.


In [ ]:
def _arm_name(universe: str, probe: str) -> str:
    return f"{universe}_{probe}"


def _parse_arm(arm: str) -> tuple[str, str]:
    universe, probe = str(arm).split("_", 1)
    return universe, probe


configs = {}
for universe in UNIVERSES:
    for probe, blend in PROBES.items():
        configs[_arm_name(universe, probe)] = config_from_stack(
            {"UNIVERSE_STAR": universe},
            hyp_overrides={
                "signal_family": "tsmom",
                "tsmom_blend": blend,
                "pair_vt": False,
                "book_vt": False,
            },
        )
register_hypothesis_arms(HYP_ID, list(configs.keys()), sleeve=SLEEVE, overwrite=True)
print("registered", list(configs.keys()))
for name, cfg in configs.items():
    print(name, "pairs=", len(cfg.pairs), "cost=", cfg.cost_profile, "blend=", cfg.tsmom_blend)


## 4. Load both universe panels

The canonical parquet is one file. Building B replaces A at that path, so this notebook loads a cache hit only when the file already holds that book; otherwise it builds in memory with `cache=False`. After freeze, §7 writes the winner to the canonical path for H-002+.


In [ ]:
def _read_parquet(path: str) -> pd.DataFrame:
    if not os.path.isfile(path):
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if df.empty:
        return df
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
    if "pair" in out.columns:
        out["pair"] = out["pair"].astype(str).str.upper()
    return out


def _load_or_build(label: str) -> pd.DataFrame:
    uni = core_universe(label)
    want = {str(p).upper() for p in uni["instruments"]}
    cached = _read_parquet(price_panel_path("1d", data_dir=DATA_DIR))
    if not cached.empty and "pair" in cached.columns and want <= set(cached["pair"].unique()):
        return cached.loc[cached["pair"].isin(want)].copy()
    warnings.warn(
        f"cache miss for universe {label}; building in memory (cache=False)"
    )
    return build_s3_price_panel(
        universe=label,
        interval="1d",
        start=IS_START,
        cache=False,
        data_dir=DATA_DIR,
    )


rates = _read_parquet(os.path.join(DATA_DIR, "g10_policy_rates.parquet"))
s1_weekly = None
s2_daily = None
for cand in (
    os.path.join(ROOT, "01_data", "data_files", "s1_equities", "s1_period_returns.parquet"),
    os.path.join(ROOT, "04_backtest", "s1_equities", "artifacts", "s1_period_returns.parquet"),
):
    if os.path.isfile(cand):
        s1 = pd.read_parquet(cand)
        if "date" in s1.columns:
            s1 = s1.set_index(pd.to_datetime(s1["date"]))
        num = s1.select_dtypes(include=[np.number])
        if not num.empty:
            s1_weekly = num.iloc[:, 0]
        break

panels = {}
panels_is = {}
split_rows = []
for label in UNIVERSES:
    panel = _load_or_build(label)
    if panel is None or panel.empty or "date" not in panel.columns:
        warnings.warn(f"empty panel for universe {label}")
        panels[label] = pd.DataFrame()
        panels_is[label] = pd.DataFrame()
        continue
    panel = panel.copy()
    panel["date"] = pd.to_datetime(panel["date"])
    panel_is, _panel_hold = split_is_oos(panel, is_end=RESEARCH_IS_END_S3)
    hold = (panel["date"] > RESEARCH_IS_END_S3) & (panel["date"] <= S3_HOLDOUT_END)
    is_n = int(panel_is["date"].nunique()) if not panel_is.empty else 0
    oos_n = int(panel.loc[hold, "date"].nunique())
    panels[label] = panel
    panels_is[label] = panel_is
    split_rows.append(
        {
            "universe": label,
            "n_instruments": int(panel["pair"].nunique()) if "pair" in panel.columns else 0,
            "is_start": panel_is["date"].min() if is_n else pd.NaT,
            "is_end": panel_is["date"].max() if is_n else pd.NaT,
            "is_sessions": is_n,
            "holdout_sessions": oos_n,
            "is_oos_ratio": f"{is_n}:{oos_n}" if oos_n else f"{is_n}:0",
            "holdout_scored": False,
        }
    )

display(pd.DataFrame(split_rows))
print("rates rows:", len(rates), "(used for FX swap on universe A)")
print("holdout 2017-2019 is sealed — dates counted, not backtested")


## 5. Fold-val + metrics

Expanding 3-fold walk-forward on **research IS only**, folds built **per universe** from that book’s session dates. Rank later on median val net Sharpe of `*_12m` only. Full-IS is reporting, not the pick. PSR / DSR are commentary.


In [ ]:
def _metric_row(name: str, res, *, fold_id: int | None = None) -> dict:
    universe, probe = _parse_arm(name)
    row = {"arm": name, "universe": universe, "probe": probe}
    if fold_id is not None:
        row["fold_id"] = fold_id
    for k in FULL_IS_COLS:
        if k == "arm":
            continue
        row[k] = res.metrics.get(k, float("nan") if k != "n_entries" else 0)
    return row


fold_df = pd.DataFrame()
full_is_df = pd.DataFrame()
returns_12m = {}
fold_parts = []
full_parts = []
n_loc = n_trials_local(configs)
n_stk = n_trials_stack(HYP_ID, configs, sleeve=SLEEVE)
print("n_trials_local", n_loc, "n_trials_stack", n_stk)

for universe in UNIVERSES:
    panel_is = panels_is.get(universe, pd.DataFrame())
    if panel_is.empty or "date" not in panel_is.columns:
        print("SKIP", universe, ": research IS panel missing or empty.")
        continue
    u_configs = {
        name: cfg for name, cfg in configs.items() if name.startswith(universe + "_")
    }
    is_dates = pd.DatetimeIndex(panel_is["date"].drop_duplicates().sort_values())
    folds = make_s3_folds(
        is_dates,
        n_folds=3,
        embargo_bars=embargo_bars_for_config(bar="1d"),
    )
    print(f"\n=== universe {universe} folds ===")
    display(fold_table(folds).assign(universe=universe))
    dates = pd.to_datetime(panel_is["date"])
    rates_arg = rates if not rates.empty else None
    for name, cfg in u_configs.items():
        for f in folds:
            res = run_s3_backtest(
                panel_is,
                cfg,
                date_mask=dates.isin(f.val_dates),
                s1_weekly=s1_weekly,
                s2_daily=s2_daily,
                rates_df=rates_arg,
                n_trials_local=n_loc,
                n_trials_stack=n_stk,
            )
            fold_parts.append(_metric_row(name, res, fold_id=f.fold_id))
        res_is = run_s3_backtest(
            panel_is,
            cfg,
            s1_weekly=s1_weekly,
            s2_daily=s2_daily,
            rates_df=rates_arg,
            n_trials_local=n_loc,
            n_trials_stack=n_stk,
        )
        row = _metric_row(name, res_is)
        row["full_is_sharpe"] = row["ann_sharpe"]
        full_parts.append(row)
        _, probe = _parse_arm(name)
        if probe == RANK_PROBE:
            returns_12m[universe] = res_is.returns

if fold_parts:
    fold_df = pd.DataFrame(fold_parts)
    display(fold_df)
if full_parts:
    full_is_df = pd.DataFrame(full_parts)

print("median_sharpe_hint (all arms, commentary only):", median_sharpe_hint(fold_df))


## 6. Evaluation tables

Locked columns: `ann_sharpe`, `ann_sharpe_gross`, `max_drawdown`, `corr_to_s1`, `corr_to_s2`, `calmar`, `psr`, `dsr_local`, `dsr_stack`, `skew`, `excess_kurtosis`, `cost_bps_per_year`, `n_days`, `n_entries`.

The **12M ranking table** is the freeze input. The all-arm selection table is diagnostic.


In [ ]:
if full_is_df.empty:
    print("no full-IS yet")
else:
    print("Full-IS (all probes)")
    display(full_is_df[[c for c in FULL_IS_COLS if c in full_is_df.columns]])

if fold_df.empty:
    print("no fold-val yet")
else:
    print("Arm selection table (median val Sharpe; no winner column)")
    display(arm_selection_table(fold_df, full_is_df))

    rank_12m = (
        fold_df.loc[fold_df["probe"] == RANK_PROBE]
        .groupby("universe", as_index=False)
        .agg(
            median_val_sharpe=("ann_sharpe", "median"),
            median_sharpe_gross=("ann_sharpe_gross", "median"),
            median_max_dd=("max_drawdown", "median"),
            median_corr_s1=("corr_to_s1", "median"),
            median_cost_bps=("cost_bps_per_year", "median"),
            n_folds=("fold_id", "nunique"),
        )
    )
    if not full_is_df.empty:
        fis = full_is_df.loc[
            full_is_df["probe"] == RANK_PROBE,
            ["universe", "ann_sharpe", "ann_sharpe_gross", "max_drawdown", "corr_to_s1"],
        ].rename(
            columns={
                "ann_sharpe": "full_is_sharpe",
                "ann_sharpe_gross": "full_is_sharpe_gross",
                "max_drawdown": "full_is_max_dd",
                "corr_to_s1": "full_is_corr_s1",
            }
        )
        rank_12m = rank_12m.merge(fis, on="universe", how="left")
    rank_12m = rank_12m.sort_values("median_val_sharpe", ascending=False)
    print("12M ranking table (freeze input)")
    display(rank_12m)


## 7. Plots

Three figures, three questions. Fold-val n=3 per box, so each box has the fold points overlaid — do not read quartiles as if n were large.

1. **Universes (decision plot)** — 12M probe only. Holding the recipe fixed, which book is tradable after costs?
2. **Probes (robustness)** — all lookbacks, grouped by universe. Does the 12M ranking flip at 3M / 1M? If it does, that is not a reason to freeze `HORIZON_STAR`.
3. **Sharpe over time** — rolling 252-day net Sharpe of the 12M probe on research IS, with calendar-year Sharpe in the lower panel. Path shape, not a second ranking statistic.


In [ ]:
def _boxes_with_points(ax, df, column, by, title, order=None):
    if order is None:
        labels = [str(x) for x in df[by].drop_duplicates().tolist()]
    else:
        labels = [x for x in order if x in set(df[by].astype(str))]
    data = [df.loc[df[by].astype(str) == lab, column].dropna().to_numpy() for lab in labels]
    ax.boxplot(data, labels=labels, widths=0.45)
    rng = np.random.default_rng(0)
    for i, lab in enumerate(labels, start=1):
        y = df.loc[df[by].astype(str) == lab, column].dropna().to_numpy()
        if y.size == 0:
            continue
        x = i + rng.normal(0.0, 0.04, size=y.size)
        ax.scatter(x, y, zorder=3, color="C0", alpha=0.85)
    ax.axhline(0.0, color="0.6", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel(by)


def _grouped_boxes(ax, df, column, *, group_col, hue_col, title, group_order=None, hue_order=None):
    if group_order is None:
        groups = [str(x) for x in df[group_col].drop_duplicates().tolist()]
    else:
        groups = [x for x in group_order if x in set(df[group_col].astype(str))]
    if hue_order is None:
        hues = [str(x) for x in df[hue_col].drop_duplicates().tolist()]
    else:
        hues = [x for x in hue_order if x in set(df[hue_col].astype(str))]
    width = 0.35
    colors = {"A": "C0", "B": "C1"}
    rng = np.random.default_rng(1)
    for j, hue in enumerate(hues):
        offset = (j - (len(hues) - 1) / 2.0) * width
        pos = [i + 1 + offset for i in range(len(groups))]
        data = [
            df.loc[(df[group_col].astype(str) == g) & (df[hue_col].astype(str) == hue), column]
            .dropna()
            .to_numpy()
            for g in groups
        ]
        bp = ax.boxplot(
            data,
            positions=pos,
            widths=width * 0.8,
            patch_artist=True,
        )
        color = colors.get(hue, f"C{j}")
        for patch in bp["boxes"]:
            patch.set_facecolor(color)
            patch.set_alpha(0.35)
        for i, g in enumerate(groups):
            y = df.loc[(df[group_col].astype(str) == g) & (df[hue_col].astype(str) == hue), column].dropna().to_numpy()
            if y.size == 0:
                continue
            x = pos[i] + rng.normal(0.0, 0.03, size=y.size)
            ax.scatter(x, y, zorder=3, color=color, label=hue if i == 0 else None)
    ax.set_xticks([i + 1 for i in range(len(groups))])
    ax.set_xticklabels(groups)
    ax.axhline(0.0, color="0.6", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, title=hue_col)


if fold_df.empty:
    print("SKIP plots: no fold-val")
else:
    rank_df = fold_df.loc[fold_df["probe"] == RANK_PROBE].copy()
    fig1, axes1 = plt.subplots(1, 2, figsize=(10, 4))
    _boxes_with_points(
        axes1[0],
        rank_df,
        "ann_sharpe",
        "universe",
        "12M fold-val net Sharpe by universe",
        order=list(UNIVERSES),
    )
    _boxes_with_points(
        axes1[1],
        rank_df,
        "max_drawdown",
        "universe",
        "12M fold-val max DD by universe",
        order=list(UNIVERSES),
    )
    fig1.suptitle(f"{HYP_ID} Figure 1 — universes (ranking probe only)")
    fig1.tight_layout()
    plt.show()

    fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4))
    _grouped_boxes(
        axes2[0],
        fold_df,
        "ann_sharpe",
        group_col="probe",
        hue_col="universe",
        title="fold-val net Sharpe by probe",
        group_order=list(PROBES.keys()),
        hue_order=list(UNIVERSES),
    )
    _grouped_boxes(
        axes2[1],
        fold_df,
        "max_drawdown",
        group_col="probe",
        hue_col="universe",
        title="fold-val max DD by probe",
        group_order=list(PROBES.keys()),
        hue_order=list(UNIVERSES),
    )
    fig2.suptitle(f"{HYP_ID} Figure 2 — probes (robustness; do not freeze HORIZON_STAR)")
    fig2.tight_layout()
    plt.show()

if not returns_12m:
    print("SKIP Sharpe-over-time: no 12M IS returns")
else:
    year_rows = []
    fig3, axes3 = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for universe, ret in returns_12m.items():
        r = pd.to_numeric(ret, errors="coerce").dropna().astype(float).sort_index()
        r = r.loc[r.index <= pd.Timestamp(RESEARCH_IS_END_S3)]
        if r.empty:
            continue
        roll = r.rolling(252, min_periods=126)
        mu = roll.mean()
        sd = roll.std(ddof=1)
        roll_sharpe = np.sqrt(PERIODS_PER_YEAR) * mu / sd
        axes3[0].plot(roll_sharpe.index, roll_sharpe.values, label=f"{universe}_12m")
        yearly = {}
        for year, s in r.groupby(r.index.year):
            yearly[int(year)] = metrics_from_returns(
                s, periods_per_year=PERIODS_PER_YEAR
            )["ann_sharpe"]
        year_index = pd.to_datetime([f"{y}-12-31" for y in yearly])
        axes3[1].plot(year_index, list(yearly.values()), marker="o", label=f"{universe}_12m")
        for year, sharpe in yearly.items():
            year_rows.append({"universe": universe, "year": int(year), "ann_sharpe": sharpe})
    axes3[0].axhline(0.0, color="0.6", lw=0.8)
    axes3[0].set_ylabel("rolling 252d net Sharpe")
    axes3[0].set_title("12M probe — rolling net Sharpe (research IS)")
    axes3[0].legend()
    axes3[1].axhline(0.0, color="0.6", lw=0.8)
    axes3[1].set_ylabel("calendar-year net Sharpe")
    axes3[1].set_title("12M probe — yearly net Sharpe (research IS)")
    axes3[1].legend()
    fig3.suptitle(f"{HYP_ID} Figure 3 — Sharpe over time (ranking probe, IS only)")
    fig3.tight_layout()
    plt.show()
    if year_rows:
        year_df = pd.DataFrame(year_rows).pivot(index="year", columns="universe", values="ann_sharpe")
        print("Calendar-year net Sharpe (12M probe, research IS)")
        display(year_df)


## 8. Freeze `UNIVERSE_STAR`

Pre-registered rule (apply after reading the 12M table and Figure 1; 1M / 3M / Figure 2 / yearly path are not inputs):

1. Winner = higher median fold-val **net** Sharpe of `*_12m`.
2. If that median is **> 0**, freeze the winner.
3. If the winner is **≤ 0** but the other book’s 12M median is **> 0**, freeze the other book.
4. If both ≤ 0, write nothing and stop the sleeve — do not default to B after seeing the test.
5. Do not re-pick after H-003+.

Then rebuild the canonical daily parquet as the winner so H-002+ load one file. Existence kill (median walk-forward net Sharpe of 12M on the frozen book) is **H-002**, not this cell.


In [ ]:
suggested = None
if fold_df.empty or "probe" not in fold_df.columns:
    print("No fold-val 12M Sharpes — leave UNIVERSE_STAR unset.")
else:
    med = (
        fold_df.loc[fold_df["probe"] == RANK_PROBE]
        .groupby("universe")["ann_sharpe"]
        .median()
        .dropna()
    )
    print("12M median fold-val net Sharpe by universe:")
    print(med)
    if not med.empty:
        ranked = med.sort_values(ascending=False)
        winner = str(ranked.index[0])
        winner_sr = float(ranked.iloc[0])
        positive = med[med > 0].sort_values(ascending=False)
        if winner_sr > 0:
            suggested = winner
            print(f"suggested UNIVERSE_STAR={suggested} (12M median {winner_sr:.3f} > 0)")
        elif not positive.empty:
            suggested = str(positive.index[0])
            print(
                f"winner {winner} median {winner_sr:.3f} <= 0; "
                f"fallback UNIVERSE_STAR={suggested} (12M median {float(positive.iloc[0]):.3f} > 0)"
            )
        else:
            print("STOP THE SLEEVE: both 12M median walk-forward net Sharpes are not > 0.")

# Override only if you reject the 12M rule above. Do not copy 1M/3M.
PICK = suggested

if PICK not in {"A", "B"}:
    print("UNIVERSE_STAR not written.")
else:
    stack = update_star_stack_key(STACK_PATH, "UNIVERSE_STAR", PICK)
    print("wrote", STACK_PATH, "UNIVERSE_STAR=", load_star_stack(STACK_PATH).get("UNIVERSE_STAR"))
    print("HORIZON_STAR stays", load_star_stack(STACK_PATH).get("HORIZON_STAR"))
    rebuilt = build_s3_price_panel(
        universe=PICK,
        interval="1d",
        start=IS_START,
        cache=True,
        data_dir=DATA_DIR,
    )
    print(
        "canonical panel rows=",
        len(rebuilt),
        "path=",
        price_panel_path("1d", data_dir=DATA_DIR),
    )


## 9. Sealed holdout stays closed

2017–2019 is counted in §4 so the split is visible. It is not a backtest input and is not used to pick `UNIVERSE_STAR`. The later OOS kill (holdout net Sharpe, SPY down-market corr) is post-freeze, not this notebook.


In [ ]:
print("Do not open 2017-2019 in H-001.")
print("IS end", RESEARCH_IS_END_S3.date(), "holdout end", S3_HOLDOUT_END.date())
print("UNIVERSE_STAR=", load_star_stack(STACK_PATH).get("UNIVERSE_STAR"))
print("HORIZON_STAR=", load_star_stack(STACK_PATH).get("HORIZON_STAR"))
print("Next: H-002 existence kill on the frozen book, 12M only.")
